# Run From Directories Example

In [ ]:
from pathlib import Path

from sqlmodel import create_engine

from cali.runner import CaliRunner
from cali.sqlmodel import (
    AnalysisSettings,
    DetectionSettings,
    Experiment,
    print_cali_results,
    save_experiment_to_database,
)

In [6]:
data_path = (
    "/Users/fdrgsp/Documents/git/cali/tests/test_data/evoked/evk.tensorstore.zarr"
)
output_path = "/Users/fdrgsp/Documents/git/cali/tests/test_data/evoked/"

In [7]:
# create a new experiment using the data in the specified directory
exp = Experiment.create_from_data(
    name="My Experiment",
    data_path=data_path,
    plate_maps={
        "genotype": {"B5": "WT"},
        "treatment": {"B5": "Vehicle"},
    },
    description=f"Experiment from {data_path}",
)
# save the experiment to a new database
save_experiment_to_database(
    exp, output_path, database_name="results_2.cali", overwrite=True
)

2025-11-24 22:12:02,580 - cali_logger - INFO - 💾 Experiment analysis updated and saved to database at /Users/fdrgsp/Documents/git/cali/tests/test_data/evoked/results_2.cali.


In [9]:
engine = create_engine(f"sqlite:///{Path(output_path) / 'results_2.cali'}")
print_cali_results(engine, show_settings=False)

📊 No analysis results found in database


In [10]:
# initialize CaliRunner
runner = CaliRunner()

In [ ]:
# specify dataset path, detection settings, and analysis settings
detection_settings = DetectionSettings(
    method="cellpose",
    model_type="custom",
    custom_model="/Users/fdrgsp/Documents/git/cali/src/cali/detection/cellpose_models/cp3_img8_epoch7000_py",
)
analysis_settings = AnalysisSettings(
    dff_window=130,
    neuropil_min_pixels=100,
    neuropil_correction_factor=0.7,
    neuropil_inner_radius=2,
)

# run analysis using new settings on existing detected ROIs (detection_id parameter)
runner.run(
    exp,
    data_path,
    detection_settings,
    analysis_settings=analysis_settings,
    global_position_indices=[0],
    output_path=output_path,
    database_name="results_2.cali",
)

2025-11-24 22:12:24,291 - cali_logger - INFO - ⚙️ Created new DetectionSettings ID 1 (method: cellpose)
2025-11-24 22:12:24,296 - cali_logger - INFO - ⚙️ Created new AnalysisSettings ID 1
2025-11-24 22:12:24,298 - cali_logger - INFO - 🔍 Running detection...
2025-11-24 22:12:27,194 - cali_logger - INFO - Use GPU: False
2025-11-24 22:12:27,194 - cali_logger - INFO - Loading model from `/Users/fdrgsp/Documents/git/cali/src/cali/detection/cellpose_models/cp3_img8_epoch7000_py`.
2025-11-24 22:12:27,281 - cali_logger - INFO - Processing 1 positions in 1 batches of 8
Running Cellpose: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]
2025-11-24 22:12:27,909 - cali_logger - INFO - ✅ Detection complete!
2025-11-24 22:12:27,911 - cali_logger - INFO - 💾 Final commit for remaining 1 FOVs
2025-11-24 22:12:27,911 - cali_logger - INFO - ✅ Detection committed: 4 ROIs across 1 FOVs
2025-11-24 22:12:27,923 - cali_logger - INFO - 📊 Created full analysis AnalysisResult ID 1 (DetectionSettings=1, AnalysisSetti

In [13]:
print_cali_results(engine, show_settings=True)

All Analysis Results (1 result)
└── 📊 Analysis Result #1
    ├── 📅 Created: 2025-11-24 22:12:27.922226
    ├── 📍 Positions Analyzed (1 position)
    │   └── Position 0
    ├── ⚙️ Detection Settings (ID: 1)
    │   ├── 📅 Created: 2025-11-24 22:12:24.275611
    │   ├── 🔬 Method: cellpose
    │   └── 🟡 Cellpose Parameters
    │       ├── Model: custom
    │       ├── Diameter: auto-detect
    │       ├── Cell prob threshold: 0.0
    │       ├── Flow threshold: 0.4
    │       ├── Min size: 10 px
    │       ├── Normalize: True
    │       └── Batch size: 8
    ├── ⚙️ Analysis Settings (ID: 1)
    │   ├── 📅 Created: 2025-11-24 22:12:24.275888
    │   ├── ✨ Experiment type: Spontaneous Activity
    │   ├── 🧵 Threads: 1
    │   ├── 🔵 Neuropil Correction
    │   │   ├── Inner radius: 0 px
    │   │   ├── Min pixels: 0
    │   │   └── Correction factor: 0.0
    │   ├── 📈 Signal Processing
    │   │   ├── ΔF/F window: 130
    │   │   └── Decay constant: 0.0
    │   ├── 🔍 Peak Detection
    │   │   ├── Height: 3.0 (multiplier)
    │   │   ├── Distance: 2 frames
    │   │   └── Prominence multiplier: 1.0
    │   ├── ⚡ Spike Detection
    │   │   └── Threshold: 1.0 (multiplier)
    │   ├── 💥 Burst Analysis
    │   │   ├── Threshold: 30.0%
    │   │   ├── Min duration: 3s
    │   │   └── Gaussian sigma: 2.0s
    │   └── 🔗 Synchrony Analysis
    │       ├── Calcium jitter window: 2
    │       ├── Network threshold: 90.0%
    │       └── Spike cross-corr lag: 5
    └── 🧪 Experiment (ID: 1)
        ├── Name: My Experiment
        ├── Description: Experiment from 
        │   /Users/fdrgsp/Documents/git/cali/tests/test_data/evoked/evk.tensorstore.zarr
        └── 📋 96-well (96-well)
            └── 🧫 B5 - 🧪 Conditions: WT, Vehicle
                └── 📷 B5_0000 (fov: 0 - pos: 0)
                    ├── 🔬 ROI 1 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 2 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 3 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    └── 🔬 ROI 4 - 🔋 active - ✨ spontaneous
                        ├── 🎭 ROI mask available
                        ├── 📊 Trace data available
                        └── 📈 Data analysis available

In [14]:
# specify dataset path, detection settings, and analysis settings
detection_settings = DetectionSettings(method="cellpose", model_type="cyto3")
analysis_settings_id = 1  # reuse analysis settings from previous run

# run analysis using new settings on existing detected ROIs (detection_id parameter)
runner.run(
    exp,
    data_path,
    detection_settings,
    analysis_settings=analysis_settings_id,
    global_position_indices=[0],
    output_path=output_path,
    database_name="results_2.cali",
)

2025-11-24 22:14:37,371 - cali_logger - INFO - ⚙️ Created new DetectionSettings ID 2 (method: cellpose)
2025-11-24 22:14:37,373 - cali_logger - INFO - ♻️ Reusing existing AnalysisSettings ID 1
2025-11-24 22:14:37,374 - cali_logger - INFO - 🔍 Running detection...
2025-11-24 22:14:37,374 - cali_logger - INFO - Use GPU: False
2025-11-24 22:14:37,375 - cali_logger - INFO - Loading model from `cyto3`.
2025-11-24 22:14:37,450 - cali_logger - INFO - Processing 1 positions in 1 batches of 8
Running Cellpose: 100%|██████████| 1/1 [00:00<00:00,  2.00it/s]
2025-11-24 22:14:37,952 - cali_logger - INFO - ✅ Detection complete!
2025-11-24 22:14:37,954 - cali_logger - INFO - 💾 Final commit for remaining 1 FOVs
2025-11-24 22:14:37,955 - cali_logger - INFO - ✅ Detection committed: 9 ROIs across 1 FOVs
2025-11-24 22:14:37,966 - cali_logger - INFO - 📊 Created full analysis AnalysisResult ID 2 (DetectionSettings=2, AnalysisSettings=1, positions=[0])
2025-11-24 22:14:37,967 - cali_logger - INFO - 📊 Running 

In [15]:
print_cali_results(engine, show_settings=False)

All Analysis Results (2 results)
├── 📊 Analysis Result #1
│   ├── 📅 Created: 2025-11-24 22:12:27.922226
│   ├── 📍 Positions Analyzed (1 position)
│   │   └── Position 0
│   ├── ⚙️ Detection Settings (ID: 1)
│   │   ├── 📅 Created: 2025-11-24 22:12:24.275611
│   │   └── 🔬 Method: cellpose
│   ├── ⚙️ Analysis Settings (ID: 1)
│   │   ├── 📅 Created: 2025-11-24 22:12:24.275888
│   │   └── ✨ Experiment type: Spontaneous Activity
│   └── 🧪 Experiment (ID: 1)
│       ├── Name: My Experiment
│       ├── Description: Experiment from 
│       │   /Users/fdrgsp/Documents/git/cali/tests/test_data/evoked/evk.tensorstore.zarr
│       └── 📋 96-well (96-well)
│           └── 🧫 B5 - 🧪 Conditions: WT, Vehicle
│               └── 📷 B5_0000 (fov: 0 - pos: 0)
│                   ├── 🔬 ROI 1 - 🔋 active - ✨ spontaneous
│                   │   ├── 🎭 ROI mask available
│                   │   ├── 📊 Trace data available
│                   │   └── 📈 Data analysis available
│                   ├── 🔬 ROI 2 - 🔋 active - ✨ spontaneous
│                   │   ├── 🎭 ROI mask available
│                   │   ├── 📊 Trace data available
│                   │   └── 📈 Data analysis available
│                   ├── 🔬 ROI 3 - 🔋 active - ✨ spontaneous
│                   │   ├── 🎭 ROI mask available
│                   │   ├── 📊 Trace data available
│                   │   └── 📈 Data analysis available
│                   └── 🔬 ROI 4 - 🔋 active - ✨ spontaneous
│                       ├── 🎭 ROI mask available
│                       ├── 📊 Trace data available
│                       └── 📈 Data analysis available
└── 📊 Analysis Result #2
    ├── 📅 Created: 2025-11-24 22:14:37.965195
    ├── 📍 Positions Analyzed (1 position)
    │   └── Position 0
    ├── ⚙️ Detection Settings (ID: 2)
    │   ├── 📅 Created: 2025-11-24 22:14:37.356241
    │   └── 🔬 Method: cellpose
    ├── ⚙️ Analysis Settings (ID: 1)
    │   ├── 📅 Created: 2025-11-24 22:12:24.275888
    │   └── ✨ Experiment type: Spontaneous Activity
    └── 🧪 Experiment (ID: 1)
        ├── Name: My Experiment
        ├── Description: Experiment from 
        │   /Users/fdrgsp/Documents/git/cali/tests/test_data/evoked/evk.tensorstore.zarr
        └── 📋 96-well (96-well)
            └── 🧫 B5 - 🧪 Conditions: WT, Vehicle
                └── 📷 B5_0000 (fov: 0 - pos: 0)
                    ├── 🔬 ROI 1 - 🪫 inactive - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 2 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 3 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 4 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 5 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 6 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 7 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Data analysis available
                    ├── 🔬 ROI 8 - 🔋 active - ✨ spontaneous
                    │   ├── 🎭 ROI mask available
                    │   ├── 📊 Trace data available
                    │   └── 📈 Dat